### 1. Zhu et al , 2018 : On the radio detectability of circumplanetary discs





In [ ]:
import numpy as np
import pandas as pd
from astropy.constants import G, M_jup, R_jup, sigma_sb

In [ ]:
# define the constants needed in appropriate cgs units 
G = G.to('cm^3 / (g s^2)')
M_jup = M_jup.to('g')
R_jup = R_jup.to('cm')
sigmaSB = sigma_sb.to('erg / (cm^2 s K^4)')

In [ ]:
# Where to store the variables is a problem especially if I am exploring a parameter space 



In [ ]:
# --- CPD Section 2 equations in Python -----------------------
# Source: Zhu, Andrews & Isella (2018) MNRAS 479, 1850 (arXiv:1708.07287)
# -------------------------------------------------------------


# --- Constants (cgs) ---
G      = 6.67430e-8        # cm^3 g^-1 s^-2
sigmaB = 5.670374419e-5    # erg cm^-2 s^-1 K^-4
kB     = 1.380649e-16      # erg K^-1
mH     = 1.6735575e-24     # g

M_jup  = 1.898e30          # g
R_jup  = 7.1492e9          # cm
L_sun  = 3.828e33          # erg/s
sec_per_yr = 3.154e7       # s

# --- ---
def omega_k(Mp, R):
    """"
    Keplerian angular velocity at radius R around a planet of mass Mp (1/s)
    Relevant to equation (5)
    Variables input: Mp (g)
    """
    return np.sqrt(G * Mp / R**3)

def T_ext(R, R_p, M_p, Mdot_p, T_ISM=10.0, mode="boundary", Lplanet=None):
    """
    External irradiation temperature T_ext (Eq. 2 in Zhu+ 2018).

    Parameters
    ----------
    R : float
        Radius (cm) where T_ext is evaluated.
    R_p : float
        Planet radius (cm).
    M_p : float
        Planet mass (g).
    Mdot_p : float
        Planet accretion rate (g/s).
    T_ISM : float, optional
        Background ISM temperature (K). Default = 10 K.
    mode : str, optional
        "boundary" -> boundary layer irradiation (L_acc).
        "planet"   -> bright planet irradiation (Lplanet must be given).
    Lplanet : float, optional
        Planet luminosity (erg/s). Required if mode="planet".

    Returns
    -------
    T_ext : float
        Effective external temperature (K).
    """
    if mode == "boundary":
        L_irr = G * M_p * Mdot_p / (2 * R_p)  # L_acc
    elif mode == "planet":
        if Lplanet is None:
            raise ValueError("For mode='planet', please supply Lplanet in erg/s")
        L_irr = Lplanet
    else:
        raise ValueError("mode must be 'boundary' or 'planet'")

    # Irradiation temperature
    T_irr = (L_irr / (4 * np.pi * sigmaB * R**2))**0.25

    # Combine with ISM background
    Text = (T_ISM**4 + T_irr**4)**0.25
    return Text

def Tc_midplane_from_Sigma(Mp, Mdot_p, Sigma, kappa_R, R, Rin, T_ext):

    """
    Midplane temperature at radius R given surface density Sigma
    Relevant to equation (4)
    Constant input: kappa_R (cm^2/g) , Rin (cm)
    Variable input: Mp (g), Mdot_p (g/s), Sigma (g/cm^2), T_ext (K)
    """
    A = (9 * G * Mp * Mdot_p * Sigma * kappa_R) / (128 * np.pi * sigmaB * R**3)
    f = (1.0 - np.sqrt(Rin / R))
    return (A * f + T_ext**4)**0.25

def Sigma_eq6(Mp, Mdot_p, alpha, kappa_R, R, Rin, mu=2.4):
    """
    Analytical surface density ignoring background irradiation and assuming viscous heating dominates
    Equivalent to equation (6) in Zhu et al. (2018)
    Constant input: Gas constant Rgas , kappa_R (cm^2/g) , Rin (cm)
    Variable input: Mp (g), Mdot_p (g/s), alpha 
    Constant input: mu (mean molecular weight, default 2.4) from Zhu et al. (2018), kappa_R (cm^2/g)
    """
    Rgas = 8.314*1e7  # erg/(mol K)  (gas constant per mole by default, NEED CONFIRM UNIT)
    #Rgas = Rgas * 1e7 / 0.0024  # erg/(g K)  (gas constant per gram for mu=2.4)
    #Rgas = kB / (mu * mH)  # erg/(g K)  (gas constant per gram, using mu) 
    term = (sigmaB * G * Mp * Mdot_p**3 / (alpha**4 * (np.pi**3) * kappa_R * R**3))**0.2
    mu_term = (mu / Rgas)**0.8
    factor = (1 - np.sqrt(Rin/R))**(3/5)
    return ((2**(7/5)) / (3**(6/5))) * term * mu_term * factor

def Sigma_eq7(Mp, Mdot_p, alpha, R, T_ext, mu=2.4):
    """
    Analytical surface density ignoring background irradiation and assuming viscous heating dominates
    Equivalent to equation (7) in Zhu et al. (2018)
    Constant input: Gas constant Rgas , kappa_R (cm^2/g) , Rin (cm)
    Variable input: Mp (g), Mdot_p (g/s), alpha 
    Constant input: mu (mean molecular weight, default 2.4) from Zhu et al. (2018), kappa_R (cm^2/g)
    """
    Rgas = 8.314*1e7  # erg/(mol K)
    Omega = omega_k(Mp, R)
    return (Mdot_p * mu * Omega) / (3*np.pi*alpha*Rgas * T_ext)  

def solve_sigma_Tc(Mp, Mdot_p, alpha, kappa_R, R, Rin, T_ext, mu=2.4):
    """Pick surface density as minimum of equation 6 and 7, then compute T_c from Eq. (4)."""
    Sigma6 = Sigma_eq6(Mp, Mdot_p, alpha, kappa_R, R, Rin, mu=mu)
    Sigma7 = Sigma_eq7(Mp, Mdot_p, alpha, R, T_ext, mu=mu)
    Sigma = min(Sigma6, Sigma7)
    Tc = Tc_midplane_from_Sigma(Mp, Mdot_p, Sigma, kappa_R, R, Rin, T_ext)
    return Sigma, Tc


def Tb_mm(Sigma, Tc, T_ext, kappa_mm):
    tau_mm = 0.5 * kappa_mm * Sigma
    if tau_mm > 0.5:
        return ((3/8) * (Tc**4) + T_ext**4)**0.25
    else:
        return 2.0 * tau_mm * Tc

def Tb_avg_over_disc(Rin, Rout, Tb_of_R):
    """
    Average brightness temperature over the disc from Rin to Rout
    Equivalent to equation (9)
    """
    Rs = np.geomspace(Rin, Rout, 256) 
    Tb_vals = Tb_of_R(Rs)  # brightness temperature at each radius
    num = np.trapz(Tb_vals * 2*np.pi*Rs, Rs)  # numerator
    den = np.pi * (Rout**2)  # denominator
    return num / den

def disc_mass(Mp, Mdot, alpha, kappa_R, Rin, Rout, T_ext):
    Rs = np.geomspace(Rin, Rout, 256)
    Sigmas = []
    for R in Rs:
        Sigma, Tc = solve_sigma_Tc(Mp, Mdot, alpha, kappa_R, R, Rin, T_ext)
        Sigmas.append(Sigma)
    integrand = 2*np.pi*Rs*np.array(Sigmas)
    Md = np.trapz(integrand, Rs)
    return Md  # grams


#Text

In [18]:

# --- Constants (cgs) ---
G      = 6.67430e-8        # cm^3 g^-1 s^-2
sigmaB = 5.670374419e-5    # erg cm^-2 s^-1 K^-4
kB     = 1.380649e-16      # erg K^-1
mH     = 1.6735575e-24     # g

M_jup  = 1.898e30          # g
R_jup  = 7.1492e9          # cm
L_sun  = 3.828e33          # erg/s
sec_per_yr = 3.154e7       # s


# --- Parameters ---
planet_masses = [1, 3, 10]    # M_J
Mdot_list     = [1e-6, 1e-7, 1e-8]  # M_J/yr
alphas        = [1e-2, 1e-3, 1e-4]

Rin  = 1.0 * R_jup
Rout = 100.0 * R_jup
T_ISM = 10.0

# Opacities
kappa_R = 10.0
kappa_mm_dict = {
    "0.87mm": 0.034 * (1.3/0.87),  # scale as λ^-β with β≈1
    "1.3mm": 0.034,
    "3mm":   0.034 * (1.3/3.0),
    "7mm":   0.034 * (1.3/7.0)
}

# Radii for Tc columns
Tc_radii = [1, 5, 10, 20, 30, 50, 100]  # in Rj

results = []

for Mp_Mj in planet_masses:
    for Mdot_Mj_yr in Mdot_list:
        for alpha in alphas:
            Mp    = Mp_Mj * M_jup
            Mdot  = Mdot_Mj_yr * M_jup / sec_per_yr

            # Disc mass
            Md = disc_mass(Mp, Mdot, alpha, kappa_R, Rin, Rout, T_ISM)
            Md_Mj = Md / M_jup

            # Accretion luminosity
            Lacc = G * Mp * Mdot / (2*Rin)
            Lacc_Lsun = Lacc / L_sun

            # Tc at specific radii
            Tc_values = []
            for rmult in Tc_radii:
                R = rmult * R_jup
                Sigma, Tc = solve_sigma_Tc(Mp, Mdot, alpha, kappa_R, R, Rin, T_ISM)
                Tc_values.append(Tc)

            # Tb averages at different wavelengths
            Tb_avgs = {}
            for lbl, kappa_mm in kappa_mm_dict.items():
                def Tb_of_R(Rs):
                    out = []
                    for r in np.atleast_1d(Rs):
                        S, T = solve_sigma_Tc(Mp, Mdot, alpha, kappa_R, r, Rin, T_ISM)
                        out.append(Tb_mm(S, T, T_ISM, kappa_mm))
                    return np.array(out)
                Tb_avgs[lbl] = Tb_avg_over_disc(Rin, Rout, Tb_of_R)

            results.append({
                "Mp [MJ]": Mp_Mj,
                "Mdot [MJ/yr]": Mdot_Mj_yr,
                "alpha": alpha,
                "Rout/RH": 0.3,  # fixed as in Zhu’s Table 1
                "Md [MJ]": Md_Mj,
                "Md/Mp": Md_Mj/Mp_Mj,
                "Lacc [Lsun]": Lacc_Lsun,
                "Rin [Rj]": 1,
                **{f"Tc {r}Rj [K]": val for r, val in zip(Tc_radii, Tc_values)},
                **{f"Tbar {lbl} [K]": Tb_avgs[lbl] for lbl in Tb_avgs}
            })

df = pd.DataFrame(results)
df

C:\Users\LHEM\AppData\Local\Temp\ipykernel_26528\1540015151.py:67: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  Md = np.trapz(integrand, Rs)
C:\Users\LHEM\AppData\Local\Temp\ipykernel_26528\1540015151.py:56: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  num = np.trapz(Tb_vals * 2*np.pi*Rs, Rs)


,Mp [MJ],Mdot [MJ/yr],alpha,Rout/RH,Md [MJ],Md/Mp,Lacc [Lsun],Rin [Rj],Tc 1Rj [K],Tc 5Rj [K],Tc 10Rj [K],Tc 20Rj [K],Tc 30Rj [K],Tc 50Rj [K],Tc 100Rj [K],Tbar 0.87mm [K],Tbar 1.3mm [K],Tbar 3mm [K],Tbar 7mm [K]
0,1,1.000000e-06,0.0100,0.3,0.000048,4.848662e-05,0.000139,1,10.0,2876.446897,1678.305497,946.263330,670.620002,431.862386,235.832471,321.450194,321.450194,316.265109,177.757994
1,1,1.000000e-06,0.0010,0.3,0.000306,3.059298e-04,0.000139,1,10.0,4558.860855,2659.934997,1499.726669,1062.861366,684.455837,373.769159,509.464043,509.464043,509.464043,509.464043
2,1,1.000000e-06,0.0001,0.3,0.001930,1.930287e-03,0.000139,1,10.0,7225.307580,4215.712983,2376.906676,1684.521814,1084.789441,592.384135,807.446012,807.446012,807.446012,807.446012
3,1,1.000000e-07,0.0100,0.3,0.000012,1.217920e-05,0.000014,1,10.0,1145.133860,668.145277,376.714252,266.978753,171.927897,93.888956,117.648293,90.362428,43.172998,18.502713
4,1,1.000000e-07,0.0010,0.3,0.000077,7.684603e-05,0.000014,1,10.0,1814.915310,1058.939104,597.051967,423.132630,272.486818,148.800744,202.822115,202.822115,205.494552,160.443322
5,1,1.000000e-07,0.0001,0.3,0.000485,4.848660e-04,0.000014,1,10.0,2876.446820,1678.305387,946.263614,670.620227,431.862479,235.832517,321.450263,321.450263,321.450263,321.450263
6,1,1.000000e-08,0.0100,0.3,0.000003,3.058374e-06,0.000001,1,10.0,455.886022,265.993586,149.973268,106.287804,68.451820,37.415177,14.887241,9.963000,4.317300,1.850271
7,1,1.000000e-08,0.0010,0.3,0.000019,1.930195e-05,0.000001,1,10.0,722.530559,421.571201,237.690750,168.452555,108.480484,59.248021,84.722885,76.515765,41.902108,18.502713
8,1,1.000000e-08,0.0001,0.3,0.000122,1.217920e-04,0.000001,1,10.0,1145.134093,668.145477,376.714336,266.978827,171.927933,93.888955,127.975098,127.975098,127.975098,129.236132
9,3,1.000000e-06,0.0100,0.3,0.000060,2.013376e-05,0.000418,1,10.0,3999.380444,2333.497665,1315.674300,932.422843,600.456698,327.898761,446.940646,446.940646,467.861019,296.116784


In [ ]:
# If it match , save these in a py file and import it
# Run it on my disk data by

2. Taylor and Adams